# Pillar N — species-distribution / habitat-suitability calibration

This notebook reproduces **N1 — Species-distribution / habitat-suitability model**. N1's exec DoD: fit a
suitability field from synthetic presences drawn from a known log-linear intensity; the fitted model's
held-out inhomogeneous-Poisson-process (IPP) log-likelihood must strictly exceed a homogeneous-null
process, and the 90% credible interval of the fitted mean intensity must cover the true intensity on
`>= 0.85` of held-out cells.

**Judgment call.** `mixle.analysis.sdm` (N1's own module: `HabitatModel`, `fit_sdm`) has not landed, so
those fall back to a reference implementation of the exact DR-ALG (Poisson-GLM MLE + Laplace covariance).
The IPP likelihood scorer and the variance recalibration are **not** fallbacks: this notebook uses the
real, already-landed `mixle.stats.processes.inhomogeneous_poisson.InhomogeneousPoissonProcessDistribution`
and `mixle.analysis.kriging.calibrate_variance`, exactly as N1's DR-ALG specifies.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# real, already-landed primitives N1's own DR-ALG is built on
from mixle.stats.processes.inhomogeneous_poisson import InhomogeneousPoissonProcessDistribution
from mixle.analysis.kriging import calibrate_variance

try:
    from mixle.analysis.sdm import HabitatModel, fit_sdm, SpeciesObservation
    HAVE_REAL_N1 = True
except ImportError:
    HAVE_REAL_N1 = False

print("using landed mixle.analysis.sdm:", HAVE_REAL_N1)

## 1. Load the committed fixture

200 cells; `X` is the design matrix (intercept + one environmental covariate), `counts` are presence counts drawn `Poisson(lambda*_c * area_c)` from a KNOWN log-linear intensity `lambda*_c = exp(a + b * env_c)` with `a=-1.0, b=2.2`; `lambda_true` is that ground truth, kept for the coverage check only.

In [ ]:
data = np.load("../../../data/pillar_validation/n_sdm_presence.npz")
X, area, counts, lambda_true, env = data["X"], data["area"], data["counts"], data["lambda_true"], data["env"]
K = X.shape[0]
print(f"{K} cells, {counts.sum()} total presences")

## 2. `fit_sdm` (reference implementation)

Maximize the exact IPP log-likelihood (scored by the real `InhomogeneousPoissonProcessDistribution._counts_log_density`, not a reimplementation of it) plus a ridge penalty; the Laplace covariance is `(X^T diag(lambda_c area_c) X + ridge I)^-1`.

In [ ]:
def neg_log_posterior(beta, X, area, counts, ridge):
    rates = np.clip(np.exp(X @ beta) * area, 1e-12, 1e8)
    dist = InhomogeneousPoissonProcessDistribution(rates, edges=np.arange(rates.size + 1))
    return -dist._counts_log_density(counts) + ridge * np.sum(beta ** 2)


if not HAVE_REAL_N1:
    class HabitatModel:
        """IC-1 `Posterior` over the fitted suitability (intensity) field."""

        def __init__(self, beta, beta_cov, design, cell_area):
            self.beta, self.beta_cov = beta, beta_cov
            self.design, self.cell_area = design, cell_area

        def samples(self, n, rng):
            L = np.linalg.cholesky(self.beta_cov + 1e-12 * np.eye(self.beta.size))
            beta_draws = self.beta[None, :] + rng.standard_normal((n, self.beta.size)) @ L.T
            return np.exp(self.design @ beta_draws.T).T * self.cell_area[None, :]

        @property
        def mean(self):
            return np.exp(self.design @ self.beta) * self.cell_area

        @property
        def cov(self):
            J = self.design * (self.mean[:, None])
            return J @ self.beta_cov @ J.T

        def credible_interval(self, level, n=4000, rng=None):
            rng = rng or np.random.default_rng(0)
            draws = self.samples(n, rng)
            a = (1 - level) / 2
            return np.quantile(draws, a, axis=0), np.quantile(draws, 1 - a, axis=0)

        def derived_quantity(self, fn, n, rng):
            from dataclasses import dataclass as _dc

            @_dc
            class _DQ:
                samples: np.ndarray
                prior_dominated: bool = False

                def credible_interval(self, level):
                    a = (1 - level) / 2
                    return np.quantile(self.samples, a, 0), np.quantile(self.samples, 1 - a, 0)

            return _DQ(samples=fn(self.samples(n, rng)))

        def critical_habitat_mask(self, threshold):
            return self.mean >= threshold


    def fit_sdm(design, area, counts, *, ridge=1e-3):
        """N1 `fit_sdm(occurrences, covariates, cell_area, *, background=None, ridge=1e-3) ->
        HabitatModel` (adapted here to take the pre-binned design/counts the fixture already provides)."""
        beta0 = np.zeros(design.shape[1])
        res = minimize(neg_log_posterior, beta0, args=(design, area, counts, ridge), method="BFGS")
        beta_hat = res.x
        lam = np.exp(design @ beta_hat) * area
        H = (design * (lam * area)[:, None]).T @ design + ridge * np.eye(design.shape[1])
        beta_cov = np.linalg.inv(H)
        return HabitatModel(beta_hat, beta_cov, design, area)

## 3. Held-out fold: fit on train cells, score on test cells

In [ ]:
split_rng = np.random.default_rng(999)
idx = split_rng.permutation(K)
n_train = int(K * 0.7)
train, test = idx[:n_train], idx[n_train:]

# The landed fit_sdm takes presence-only OCCURRENCE records (it bins them itself) rather than
# per-cell counts, and needs the species it is fitting. Expand the count vector into one record
# per presence, indexed by position within the training subset, which is the cell index fit_sdm
# resolves against the covariate rows it was handed.
train_counts = counts[train]
occurrences = [
    SpeciesObservation(species_id="focal", detection=True, location=np.array([float(cell)]))
    for cell, n in enumerate(train_counts)
    for _ in range(int(n))
]
model = fit_sdm(occurrences, X[train], area[train], species_id="focal", ridge=1e-3)
# beta/beta_cov are estimated from the training cells only; the fitted HabitatModel then projects that
# intensity field over the FULL grid (train + held-out test cells) -- that projection is exactly what the
# held-out coverage check below queries.
# fit_sdm's design carries an intercept column, so beta is one longer than X's width; project
# with the same [1, X] layout it was fitted under.
design_full = np.column_stack([np.ones(K), X])
model.design, model.cell_area = design_full, area
print("fitted beta:", np.round(model.beta, 3))

## 4. Held-out IPP log-likelihood: fitted model vs. a homogeneous null

Both log-likelihoods are scored by the real `InhomogeneousPoissonProcessDistribution` -- the fitted model must explain the held-out presences strictly better than a constant-rate null process.

In [ ]:
lam_test_fit = np.exp(design_full[test] @ model.beta) * area[test]
dist_fit = InhomogeneousPoissonProcessDistribution(np.clip(lam_test_fit, 1e-12, None), edges=np.arange(test.size + 1))
ll_fit = dist_fit._counts_log_density(counts[test])

null_rate = counts[train].sum() / area[train].sum()
lam_test_null = np.full(test.size, null_rate) * area[test]
dist_null = InhomogeneousPoissonProcessDistribution(np.clip(lam_test_null, 1e-12, None), edges=np.arange(test.size + 1))
ll_null = dist_null._counts_log_density(counts[test])

beats_null = ll_fit > ll_null
print(f"held-out log-likelihood: fitted={ll_fit:.2f}  homogeneous-null={ll_null:.2f}  beats_null={beats_null}")

## 5. Coverage of the true intensity on held-out cells + kriging-style variance recalibration

`calibrate_variance` is the real `mixle.analysis.kriging` helper -- the same generic GP/kriging-variance recalibration N1's DR-ALG calls out, reused here rather than reimplemented.

In [ ]:
cov_rng = np.random.default_rng(11)
# the landed interval is analytic (lognormal delta method), so it takes no draw count or rng
lo, hi = model.credible_interval(0.9)
covered = (lambda_true[test] >= lo[test]) & (lambda_true[test] <= hi[test])
coverage = float(covered.mean())

mean_pred = model.mean[test]
draws_test = model.samples(4000, np.random.default_rng(11))[:, test]
var_pred = draws_test.var(axis=0)
resid = lambda_true[test] - mean_pred
calibration_multiplier = calibrate_variance(var_pred, resid, target=0.9)

print(f"90% CI coverage on {test.size} held-out cells: {coverage:.3f}")
print(f"kriging-style calibration multiplier: {calibration_multiplier:.4f}")

## 6. Calibration / UQ plot: suitability with credible band vs. truth on held-out cells

In [ ]:
order = np.argsort(env[test])
fig, ax = plt.subplots(figsize=(7, 4))
ax.fill_between(env[test][order], lo[test][order], hi[test][order], color="#2E86AB", alpha=0.25, label="90% CI")
ax.plot(env[test][order], model.mean[test][order], color="#2E86AB", label="fitted mean intensity")
ax.plot(env[test][order], lambda_true[test][order], "o", ms=4, color="#C0392B", label="true intensity")
ax.set_xlabel("environmental covariate"); ax.set_ylabel("intensity (suitability)")
ax.set_title(f"held-out suitability calibration (coverage={coverage:.2f})"); ax.legend()
plt.tight_layout(); plt.show()

## 7. Definition of Done

In [ ]:
assert beats_null, f"fitted held-out log-likelihood {ll_fit:.2f} did not beat the null {ll_null:.2f}"
assert coverage >= 0.85, f"90% CI coverage {coverage:.3f} < 0.85"
print(f"PASS -- held-out log-lik fitted={ll_fit:.2f} > null={ll_null:.2f}; coverage={coverage:.3f} >= 0.85")